Goal is to inspect and prepare the dataset fro river flow  file before doing any claculation or analysis


In [ ]:
# we done this to all the datasets because all work is getting saved into the drive

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import zipfile

DATA_RAW = Path("/content/drive/MyDrive/STAT596/Project596_datafiles")
DATA_PROCESSED = DATA_RAW / "processed_data"
FIGURES = DATA_RAW / "figures"
TABLES = DATA_RAW / "tables"
NOTEBOOKS = DATA_RAW / "notebooks"

DATA_PROCESSED.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
TABLES.mkdir(exist_ok=True)
NOTEBOOKS.mkdir(exist_ok=True)

print("Project folder:", DATA_RAW)
print("Project folder exists:", DATA_RAW.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/STAT596/Project596_datafiles
Project folder exists: True


In [ ]:


river_zip_path = DATA_RAW / "tijuana_river_discharge_daily_usibwc_2023_2026.zip"

print("River ZIP path:", river_zip_path)
print("File exists:", river_zip_path.exists())

if river_zip_path.exists():
    with zipfile.ZipFile(river_zip_path, "r") as z:
        print("\nFiles inside ZIP:")
        for name in z.namelist():
            print(name)
else:
    print("\nFile was not found with this exact name.")
    print("Files in project folder that mention 'tijuana' or 'river':")
    for file in DATA_RAW.iterdir():
        if "tijuana" in file.name.lower() or "river" in file.name.lower():
            print(file.name)

River ZIP path: /content/drive/MyDrive/STAT596/Project596_datafiles/tijuana_river_discharge_daily_usibwc_2023_2026.zip
File exists: True

Files inside ZIP:
DataSetExport-Discharge.Daily Rounded@11013300-Aggregate-m^3 s-20260504004239.csv


In [ ]:
# since we comfirmed that we have the file not lets look at the structure
# and see if we need to modify anything and how its will help push the project

import io

with zipfile.ZipFile(river_zip_path, "r") as z:
    csv_name = [name for name in z.namelist() if name.endswith(".csv")][0]
    raw_text = z.open(csv_name).read().decode("utf-8-sig")

lines = raw_text.splitlines()

print("CSV file name inside ZIP:")
print(csv_name)

print("\nFirst 15 lines of raw CSV:")
for i, line in enumerate(lines[:15]):
    print(f"{i}: {line}")

print("\nLast 10 lines of raw CSV:")
for i, line in enumerate(lines[-10:]):
    print(f"{len(lines)-10+i}: {line}")


CSV file name inside ZIP:
DataSetExport-Discharge.Daily Rounded@11013300-Aggregate-m^3 s-20260504004239.csv

First 15 lines of raw CSV:
0: #Data Set Export - Discharge.Daily Rounded@11013300,,,,
1: Start of Interval (UTC-08:00),End of Interval (UTC-08:00),Value (m^3/s),Grade Code,Interpolation Type
2: 2023-01-01 00:00:00,2023-01-02 00:00:00,78.3,-1,2
3: 2023-01-02 00:00:00,2023-01-03 00:00:00,9.46,-1,2
4: 2023-01-03 00:00:00,2023-01-04 00:00:00,5.09,-1,2
5: 2023-01-04 00:00:00,2023-01-05 00:00:00,3.52,-1,2
6: 2023-01-05 00:00:00,2023-01-06 00:00:00,3.76,-1,2
7: 2023-01-06 00:00:00,2023-01-07 00:00:00,2.7,-1,2
8: 2023-01-07 00:00:00,2023-01-08 00:00:00,2.85,-1,2
9: 2023-01-08 00:00:00,2023-01-09 00:00:00,2.4,-1,2
10: 2023-01-09 00:00:00,2023-01-10 00:00:00,2.26,-1,2
11: 2023-01-10 00:00:00,2023-01-11 00:00:00,6.64,-1,2
12: 2023-01-11 00:00:00,2023-01-12 00:00:00,2.83,-1,2
13: 2023-01-12 00:00:00,2023-01-13 00:00:00,2.42,-1,2
14: 2023-01-13 00:00:00,2023-01-14 00:00:00,2.3,-1,2

Last 10 

In [ ]:

# we need to find the header, standardize columns and
# converts dates, and sorts records to match the other datasets

# Find the actual data header row
header_idx = None
for i, line in enumerate(lines):
    if line.startswith("Start of Interval"):
        header_idx = i
        break

print("Header row index:", header_idx)

# Keep only the CSV table starting from the real header
table_text = "\n".join(lines[header_idx:])

river_daily_raw = pd.read_csv(io.StringIO(table_text))

# Standardize column names
river_daily_raw.columns = (
    river_daily_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_per_", regex=False)
)

print("Original cleaned column names:")
print(river_daily_raw.columns.tolist())

# Rename  columns
rename_map = {
    "start_of_interval_utc-08:00": "start_datetime",
    "end_of_interval_utc-08:00": "end_datetime",
    "value_m^3_per_s": "discharge_m3s",
    "grade_code": "grade_code",
    "interpolation_type": "interpolation_type"
}

river_daily_raw = river_daily_raw.rename(columns=rename_map)

# Convert key fields
river_daily_raw["start_datetime"] = pd.to_datetime(river_daily_raw["start_datetime"], errors="coerce")
river_daily_raw["end_datetime"] = pd.to_datetime(river_daily_raw["end_datetime"], errors="coerce")
river_daily_raw["date"] = river_daily_raw["start_datetime"].dt.date
river_daily_raw["date"] = pd.to_datetime(river_daily_raw["date"])
river_daily_raw["discharge_m3s"] = pd.to_numeric(river_daily_raw["discharge_m3s"], errors="coerce")

# Drop rows that are not actual data records
river_daily_raw = river_daily_raw.dropna(subset=["date", "discharge_m3s"])

# Sort chronologically
river_daily_raw = river_daily_raw.sort_values("date").reset_index(drop=True)

print("Rows:", len(river_daily_raw))
print("Columns:", river_daily_raw.shape[1])
print("Date range:", river_daily_raw["date"].min(), "to", river_daily_raw["date"].max())
print("Missing discharge values:", river_daily_raw["discharge_m3s"].isna().sum())

display(river_daily_raw.head())
display(river_daily_raw.tail())

Header row index: 1
Original cleaned column names:
['start_of_interval_utc-08:00', 'end_of_interval_utc-08:00', 'value_m^3_per_s', 'grade_code', 'interpolation_type']
Rows: 1159
Columns: 6
Date range: 2023-01-01 00:00:00 to 2026-03-04 00:00:00
Missing discharge values: 0


,start_datetime,end_datetime,discharge_m3s,grade_code,interpolation_type,date
0,2023-01-01,2023-01-02,78.30,-1.0,2.0,2023-01-01
1,2023-01-02,2023-01-03,9.46,-1.0,2.0,2023-01-02
2,2023-01-03,2023-01-04,5.09,-1.0,2.0,2023-01-03
3,2023-01-04,2023-01-05,3.52,-1.0,2.0,2023-01-04
4,2023-01-05,2023-01-06,3.76,-1.0,2.0,2023-01-05


,start_datetime,end_datetime,discharge_m3s,grade_code,interpolation_type,date
1154,2026-02-28,2026-03-01,1.88,-1.0,2.0,2026-02-28
1155,2026-03-01,2026-03-02,1.85,-1.0,2.0,2026-03-01
1156,2026-03-02,2026-03-03,1.82,-1.0,2.0,2026-03-02
1157,2026-03-03,2026-03-04,1.81,-1.0,2.0,2026-03-03
1158,2026-03-04,2026-03-05,1.73,-1.0,2.0,2026-03-04


In [ ]:
# now lets check for missing dates ,duplications and comfirm the right interval
# for the project

river_daily = river_daily_raw.copy()

# Create expected daily date range
expected_dates = pd.date_range(
    start=river_daily["date"].min(),
    end=river_daily["date"].max(),
    freq="D"
)

actual_dates = pd.DatetimeIndex(river_daily["date"].dropna().unique())

missing_dates = expected_dates.difference(actual_dates)
duplicate_dates = river_daily[river_daily["date"].duplicated(keep=False)].sort_values("date")

# Check interval length in hours
river_daily["interval_hours"] = (
    river_daily["end_datetime"] - river_daily["start_datetime"]
).dt.total_seconds() / 3600

print("Expected daily dates:", len(expected_dates))
print("Actual unique dates:", len(actual_dates))
print("Missing dates:", len(missing_dates))
print("Duplicate date rows:", len(duplicate_dates))

print("\nInterval hour summary:")
display(river_daily["interval_hours"].describe())

print("\nInterval hour counts:")
display(river_daily["interval_hours"].value_counts().sort_index())

if len(missing_dates) > 0:
    print("\nFirst 10 missing dates:")
    print(missing_dates[:10].tolist())

if len(duplicate_dates) > 0:
    print("\nDuplicate date rows:")
    display(duplicate_dates)


Expected daily dates: 1159
Actual unique dates: 1159
Missing dates: 0
Duplicate date rows: 0

Interval hour summary:


,interval_hours
count,1159.0
mean,24.0
std,0.0
min,24.0
25%,24.0
50%,24.0
75%,24.0
max,24.0



Interval hour counts:


,count
interval_hours,
24.0,1159


In [ ]:
# everything looks great now we have to add
# discharge in cfs and MGD, plus year/month fields and source metadata.
# need to keep the variable "discharge_m3s" as the main unit

river_daily["discharge_cfs"] = river_daily["discharge_m3s"] * 35.3147

# Convert cubic meters per second to million US gallons per day.
# 1 m3/s = about 22.8245 million gallons per day.
river_daily["discharge_mgd"] = river_daily["discharge_m3s"] * 22.8245

river_daily["year"] = river_daily["date"].dt.year
river_daily["month"] = river_daily["date"].dt.month
river_daily["month_name"] = river_daily["date"].dt.month_name()

river_daily["site_id"] = "11013300"
river_daily["site_name"] = "Tijuana River At International Boundary"
river_daily["parameter"] = "Discharge"
river_daily["source"] = "USIBWC Water Data Portal"

# I want to stay consistant with order for readability
river_daily = river_daily[
    [
        "date",
        "start_datetime",
        "end_datetime",
        "interval_hours",
        "year",
        "month",
        "month_name",
        "site_id",
        "site_name",
        "parameter",
        "source",
        "discharge_m3s",
        "discharge_cfs",
        "discharge_mgd",
        "grade_code",
        "interpolation_type"
    ]
]

print("Rows:", len(river_daily))
print("Columns:", river_daily.shape[1])

print("\nDischarge summary in cubic meters per second:")
display(river_daily["discharge_m3s"].describe())

print("\nDischarge summary in million gallons per day:")
display(river_daily["discharge_mgd"].describe())

display(river_daily.head())
display(river_daily.tail())

Rows: 1159
Columns: 16

Discharge summary in cubic meters per second:


,discharge_m3s
count,1159.000000
mean,3.964288
std,13.506383
min,0.000000
25%,0.460000
50%,1.750000
75%,2.660000
max,253.000000



Discharge summary in million gallons per day:


,discharge_mgd
count,1159.000000
mean,90.482896
std,308.276439
min,0.000000
25%,10.499270
50%,39.942875
75%,60.713170
max,5774.598500


,date,start_datetime,end_datetime,interval_hours,year,month,month_name,site_id,site_name,parameter,source,discharge_m3s,discharge_cfs,discharge_mgd,grade_code,interpolation_type
0,2023-01-01,2023-01-01,2023-01-02,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,78.30,2765.141010,1787.158350,-1.0,2.0
1,2023-01-02,2023-01-02,2023-01-03,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,9.46,334.077062,215.919770,-1.0,2.0
2,2023-01-03,2023-01-03,2023-01-04,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,5.09,179.751823,116.176705,-1.0,2.0
3,2023-01-04,2023-01-04,2023-01-05,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,3.52,124.307744,80.342240,-1.0,2.0
4,2023-01-05,2023-01-05,2023-01-06,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,3.76,132.783272,85.820120,-1.0,2.0


,date,start_datetime,end_datetime,interval_hours,year,month,month_name,site_id,site_name,parameter,source,discharge_m3s,discharge_cfs,discharge_mgd,grade_code,interpolation_type
1154,2026-02-28,2026-02-28,2026-03-01,24.0,2026,2,February,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,1.88,66.391636,42.910060,-1.0,2.0
1155,2026-03-01,2026-03-01,2026-03-02,24.0,2026,3,March,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,1.85,65.332195,42.225325,-1.0,2.0
1156,2026-03-02,2026-03-02,2026-03-03,24.0,2026,3,March,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,1.82,64.272754,41.540590,-1.0,2.0
1157,2026-03-03,2026-03-03,2026-03-04,24.0,2026,3,March,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,1.81,63.919607,41.312345,-1.0,2.0
1158,2026-03-04,2026-03-04,2026-03-05,24.0,2026,3,March,11013300,Tijuana River At International Boundary,Discharge,USIBWC Water Data Portal,1.73,61.094431,39.486385,-1.0,2.0


In [ ]:
# i need a high flow and a seprated flow categories
# for that i need to calculate 75th, 90th, and 95th percentile discharge thresholds
# which helps identify the high flows

flow_75th = river_daily["discharge_m3s"].quantile(0.75)
flow_90th = river_daily["discharge_m3s"].quantile(0.90)
flow_95th = river_daily["discharge_m3s"].quantile(0.95)

# Create high-flow indicator variables
river_daily["high_flow_75th"] = (river_daily["discharge_m3s"] >= flow_75th).astype(int)
river_daily["high_flow_90th"] = (river_daily["discharge_m3s"] >= flow_90th).astype(int)
river_daily["extreme_flow_95th"] = (river_daily["discharge_m3s"] >= flow_95th).astype(int)

# Main high-flow day flag for later joins
# Here, high_flow_day uses the 75th percentile threshold.
river_daily["high_flow_day"] = river_daily["high_flow_75th"]

# Create descriptive flow categories
conditions = [
    river_daily["discharge_m3s"] >= flow_95th,
    river_daily["discharge_m3s"] >= flow_90th,
    river_daily["discharge_m3s"] >= flow_75th
]

choices = [
    "Extreme flow",
    "Very high flow",
    "High flow"
]

river_daily["flow_category"] = np.select(
    conditions,
    choices,
    default="Normal / lower flow"
)

print("Flow thresholds based on daily discharge_m3s:")
print("75th percentile:", round(flow_75th, 3), "m3/s")
print("90th percentile:", round(flow_90th, 3), "m3/s")
print("95th percentile:", round(flow_95th, 3), "m3/s")

print("\nHigh-flow flag counts:")
print("High flow 75th days:", river_daily["high_flow_75th"].sum())
print("High flow 90th days:", river_daily["high_flow_90th"].sum())
print("Extreme flow 95th days:", river_daily["extreme_flow_95th"].sum())

print("\nFlow category counts:")
display(river_daily["flow_category"].value_counts())

display(
    river_daily[
        [
            "date",
            "discharge_m3s",
            "discharge_mgd",
            "high_flow_75th",
            "high_flow_90th",
            "extreme_flow_95th",
            "high_flow_day",
            "flow_category"
        ]
    ].head(10)
)

Flow thresholds based on daily discharge_m3s:
75th percentile: 2.66 m3/s
90th percentile: 5.154 m3/s
95th percentile: 11.06 m3/s

High-flow flag counts:
High flow 75th days: 291
High flow 90th days: 116
Extreme flow 95th days: 58

Flow category counts:


,count
flow_category,
Normal / lower flow,868
High flow,175
Very high flow,58
Extreme flow,58


,date,discharge_m3s,discharge_mgd,high_flow_75th,high_flow_90th,extreme_flow_95th,high_flow_day,flow_category
0,2023-01-01,78.30,1787.158350,1,1,1,1,Extreme flow
1,2023-01-02,9.46,215.919770,1,1,0,1,Very high flow
2,2023-01-03,5.09,116.176705,1,0,0,1,High flow
3,2023-01-04,3.52,80.342240,1,0,0,1,High flow
4,2023-01-05,3.76,85.820120,1,0,0,1,High flow
5,2023-01-06,2.70,61.626150,1,0,0,1,High flow
6,2023-01-07,2.85,65.049825,1,0,0,1,High flow
7,2023-01-08,2.40,54.778800,0,0,0,0,Normal / lower flow
8,2023-01-09,2.26,51.583370,0,0,0,0,Normal / lower flow
9,2023-01-10,6.64,151.554680,1,1,0,1,Very high flow


In [ ]:
#create river-flow timing variables that can later be compared with rainfall and
# beach closure activity.

river_daily["flow_lag1_m3s"] = river_daily["discharge_m3s"].shift(1)

# Rolling mean discharge windows
# These include the current day and previous days.
river_daily["flow_3day_mean_m3s"] = (
    river_daily["discharge_m3s"]
    .rolling(window=3, min_periods=1)
    .mean()
)

river_daily["flow_7day_mean_m3s"] = (
    river_daily["discharge_m3s"]
    .rolling(window=7, min_periods=1)
    .mean()
)

# Recent high-flow windows based on the main 75th-percentile high_flow_day flag.
# These answer: was this date in or shortly after a high-flow period?
river_daily["high_flow_1day_window"] = (
    river_daily["high_flow_day"]
    .rolling(window=2, min_periods=1)
    .max()
    .astype(int)
)

river_daily["high_flow_3day_window"] = (
    river_daily["high_flow_day"]
    .rolling(window=4, min_periods=1)
    .max()
    .astype(int)
)

river_daily["high_flow_7day_window"] = (
    river_daily["high_flow_day"]
    .rolling(window=8, min_periods=1)
    .max()
    .astype(int)
)

print("Missing values after creating time-window variables:")
print(
    river_daily[
        [
            "flow_lag1_m3s",
            "flow_3day_mean_m3s",
            "flow_7day_mean_m3s",
            "high_flow_1day_window",
            "high_flow_3day_window",
            "high_flow_7day_window"
        ]
    ].isna().sum()
)

print("\nHigh-flow window counts:")
print("Same day or previous 1 day:", river_daily["high_flow_1day_window"].sum())
print("Same day or previous 3 days:", river_daily["high_flow_3day_window"].sum())
print("Same day or previous 7 days:", river_daily["high_flow_7day_window"].sum())

display(
    river_daily[
        [
            "date",
            "discharge_m3s",
            "flow_lag1_m3s",
            "flow_3day_mean_m3s",
            "flow_7day_mean_m3s",
            "high_flow_day",
            "high_flow_1day_window",
            "high_flow_3day_window",
            "high_flow_7day_window",
            "flow_category"
        ]
    ].head(15)
)


Missing values after creating time-window variables:
flow_lag1_m3s            1
flow_3day_mean_m3s       0
flow_7day_mean_m3s       0
high_flow_1day_window    0
high_flow_3day_window    0
high_flow_7day_window    0
dtype: int64

High-flow window counts:
Same day or previous 1 day: 327
Same day or previous 3 days: 383
Same day or previous 7 days: 464


,date,discharge_m3s,flow_lag1_m3s,flow_3day_mean_m3s,flow_7day_mean_m3s,high_flow_day,high_flow_1day_window,high_flow_3day_window,high_flow_7day_window,flow_category
0,2023-01-01,78.30,NaN,78.300000,78.300000,1,1,1,1,Extreme flow
1,2023-01-02,9.46,78.30,43.880000,43.880000,1,1,1,1,Very high flow
2,2023-01-03,5.09,9.46,30.950000,30.950000,1,1,1,1,High flow
3,2023-01-04,3.52,5.09,6.023333,24.092500,1,1,1,1,High flow
4,2023-01-05,3.76,3.52,4.123333,20.026000,1,1,1,1,High flow
5,2023-01-06,2.70,3.76,3.326667,17.138333,1,1,1,1,High flow
6,2023-01-07,2.85,2.70,3.103333,15.097143,1,1,1,1,High flow
7,2023-01-08,2.40,2.85,2.650000,4.254286,0,1,1,1,Normal / lower flow
8,2023-01-09,2.26,2.40,2.503333,3.225714,0,0,1,1,Normal / lower flow
9,2023-01-10,6.64,2.26,3.766667,3.447143,1,1,1,1,Very high flow


In [ ]:
# I am planning to structure the project with huge events so i want to
# reference those event in the data like in June 2 2023
# i need to be able to match dates and places for analysis later on

emergency_date = pd.Timestamp("2023-06-27")
emergency_window_start = pd.Timestamp("2023-05-28")
emergency_window_end = pd.Timestamp("2023-07-27")
h2s_overlap_start = pd.Timestamp("2024-09-25")

river_daily["days_from_emergency_declaration"] = (
    river_daily["date"] - emergency_date
).dt.days

river_daily["post_emergency"] = (
    river_daily["date"] >= emergency_date
).astype(int)

river_daily["emergency_reference_window"] = (
    river_daily["date"].between(emergency_window_start, emergency_window_end)
).astype(int)

river_daily["h2s_overlap_period"] = (
    river_daily["date"] >= h2s_overlap_start
).astype(int)

conditions = [
    river_daily["date"].between(emergency_window_start, emergency_window_end),
    river_daily["date"] < emergency_window_start,
    river_daily["date"] > emergency_window_end
]

choices = [
    "Emergency reference window",
    "Before emergency reference window",
    "After emergency reference window"
]

river_daily["analysis_period"] = np.select(
    conditions,
    choices,
    default="Unclassified"
)

print("Emergency declaration date:", emergency_date.date())
print("Emergency reference window:", emergency_window_start.date(), "to", emergency_window_end.date())

print("\nAnalysis period counts:")
display(river_daily["analysis_period"].value_counts())

print("\nPost-emergency counts:")
display(river_daily["post_emergency"].value_counts())

print("\nH2S overlap period counts:")
display(river_daily["h2s_overlap_period"].value_counts())

display(
    river_daily[
        [
            "date",
            "discharge_m3s",
            "high_flow_day",
            "high_flow_3day_window",
            "days_from_emergency_declaration",
            "post_emergency",
            "emergency_reference_window",
            "h2s_overlap_period",
            "analysis_period"
        ]
    ].head(10)
)

Emergency declaration date: 2023-06-27
Emergency reference window: 2023-05-28 to 2023-07-27

Analysis period counts:


,count
analysis_period,
After emergency reference window,951
Before emergency reference window,147
Emergency reference window,61



Post-emergency counts:


,count
post_emergency,
1,982
0,177



H2S overlap period counts:


,count
h2s_overlap_period,
0,633
1,526


,date,discharge_m3s,high_flow_day,high_flow_3day_window,days_from_emergency_declaration,post_emergency,emergency_reference_window,h2s_overlap_period,analysis_period
0,2023-01-01,78.30,1,1,-177,0,0,0,Before emergency reference window
1,2023-01-02,9.46,1,1,-176,0,0,0,Before emergency reference window
2,2023-01-03,5.09,1,1,-175,0,0,0,Before emergency reference window
3,2023-01-04,3.52,1,1,-174,0,0,0,Before emergency reference window
4,2023-01-05,3.76,1,1,-173,0,0,0,Before emergency reference window
5,2023-01-06,2.70,1,1,-172,0,0,0,Before emergency reference window
6,2023-01-07,2.85,1,1,-171,0,0,0,Before emergency reference window
7,2023-01-08,2.40,0,1,-170,0,0,0,Before emergency reference window
8,2023-01-09,2.26,0,1,-169,0,0,0,Before emergency reference window
9,2023-01-10,6.64,1,1,-168,0,0,0,Before emergency reference window


In [ ]:
# lets create a summary table to use later on the project and data analysis

river_overall_summary = pd.DataFrame({
    "metric": [
        "start_date",
        "end_date",
        "total_daily_records",
        "missing_discharge_values",
        "mean_discharge_m3s",
        "median_discharge_m3s",
        "max_discharge_m3s",
        "mean_discharge_mgd",
        "max_discharge_mgd",
        "high_flow_75th_days",
        "high_flow_90th_days",
        "extreme_flow_95th_days"
    ],
    "value": [
        river_daily["date"].min().date(),
        river_daily["date"].max().date(),
        len(river_daily),
        river_daily["discharge_m3s"].isna().sum(),
        round(river_daily["discharge_m3s"].mean(), 3),
        round(river_daily["discharge_m3s"].median(), 3),
        round(river_daily["discharge_m3s"].max(), 3),
        round(river_daily["discharge_mgd"].mean(), 3),
        round(river_daily["discharge_mgd"].max(), 3),
        int(river_daily["high_flow_75th"].sum()),
        int(river_daily["high_flow_90th"].sum()),
        int(river_daily["extreme_flow_95th"].sum())
    ]
})

print("Overall river-flow dataset summary:")
display(river_overall_summary)

# Flow category summary
river_category_summary = (
    river_daily
    .groupby("flow_category", as_index=False)
    .agg(
        days=("date", "count"),
        mean_discharge_m3s=("discharge_m3s", "mean"),
        median_discharge_m3s=("discharge_m3s", "median"),
        max_discharge_m3s=("discharge_m3s", "max"),
        mean_discharge_mgd=("discharge_mgd", "mean")
    )
)

river_category_summary["percent_of_days"] = (
    river_category_summary["days"] / len(river_daily) * 100
).round(2)

numeric_cols = [
    "mean_discharge_m3s",
    "median_discharge_m3s",
    "max_discharge_m3s",
    "mean_discharge_mgd"
]

river_category_summary[numeric_cols] = river_category_summary[numeric_cols].round(3)

print("\nFlow category summary:")
display(river_category_summary.sort_values("days", ascending=False))

# Year-level summary
river_year_summary = (
    river_daily
    .groupby("year", as_index=False)
    .agg(
        days=("date", "count"),
        mean_discharge_m3s=("discharge_m3s", "mean"),
        max_discharge_m3s=("discharge_m3s", "max"),
        high_flow_days=("high_flow_day", "sum"),
        very_high_flow_days=("high_flow_90th", "sum"),
        extreme_flow_days=("extreme_flow_95th", "sum")
    )
)

river_year_summary[
    ["mean_discharge_m3s", "max_discharge_m3s"]
] = river_year_summary[
    ["mean_discharge_m3s", "max_discharge_m3s"]
].round(3)

print("\nYear-level river-flow summary:")
display(river_year_summary)

# Top daily discharge records
print("\nTop 10 daily discharge days:")
display(
    river_daily[
        [
            "date",
            "discharge_m3s",
            "discharge_mgd",
            "flow_category",
            "high_flow_day",
            "high_flow_3day_window",
            "analysis_period"
        ]
    ]
    .sort_values("discharge_m3s", ascending=False)
    .head(10)
)

Overall river-flow dataset summary:


,metric,value
0,start_date,2023-01-01
1,end_date,2026-03-04
2,total_daily_records,1159
3,missing_discharge_values,0
4,mean_discharge_m3s,3.964
5,median_discharge_m3s,1.75
6,max_discharge_m3s,253.0
7,mean_discharge_mgd,90.483
8,max_discharge_mgd,5774.599
9,high_flow_75th_days,291



Flow category summary:


,flow_category,days,mean_discharge_m3s,median_discharge_m3s,max_discharge_m3s,mean_discharge_mgd,percent_of_days
2,Normal / lower flow,868,1.148,0.93,2.65,26.209,74.89
1,High flow,175,3.571,3.48,5.15,81.512,15.10
0,Extreme flow,58,43.898,26.10,253.00,1001.956,5.00
3,Very high flow,58,7.359,6.95,11.00,167.973,5.00



Year-level river-flow summary:


,year,days,mean_discharge_m3s,max_discharge_m3s,high_flow_days,very_high_flow_days,extreme_flow_days
0,2023,365,5.208,253.0,125,54,24
1,2024,366,4.230,137.0,126,39,17
2,2025,365,1.858,85.8,29,16,11
3,2026,63,7.418,159.0,11,7,6



Top 10 daily discharge days:


,date,discharge_m3s,discharge_mgd,flow_category,high_flow_day,high_flow_3day_window,analysis_period
15,2023-01-16,253.0,5774.59850,Extreme flow,1,1,Before emergency reference window
1096,2026-01-01,159.0,3629.09550,Extreme flow,1,1,After emergency reference window
386,2024-01-22,137.0,3126.95650,Extreme flow,1,1,After emergency reference window
401,2024-02-06,133.0,3035.65850,Extreme flow,1,1,After emergency reference window
402,2024-02-07,95.4,2177.45730,Extreme flow,1,1,After emergency reference window
14,2023-01-15,90.8,2072.46460,Extreme flow,1,1,Before emergency reference window
802,2025-03-13,85.8,1958.34210,Extreme flow,1,1,After emergency reference window
79,2023-03-21,83.4,1903.56330,Extreme flow,1,1,Before emergency reference window
0,2023-01-01,78.3,1787.15835,Extreme flow,1,1,Before emergency reference window
231,2023-08-20,77.2,1762.05140,Extreme flow,1,1,After emergency reference window


In [ ]:
# lets save our summary for later use

river_overall_summary_path = TABLES / "river_flow_overall_summary.csv"
river_category_summary_path = TABLES / "river_flow_category_summary.csv"
river_year_summary_path = TABLES / "river_flow_year_summary.csv"

river_overall_summary.to_csv(river_overall_summary_path, index=False)
river_category_summary.to_csv(river_category_summary_path, index=False)
river_year_summary.to_csv(river_year_summary_path, index=False)

print("Saved river-flow summary tables:")
print(river_overall_summary_path)
print(river_category_summary_path)
print(river_year_summary_path)

print("\nFile exists check:")
print("Overall summary:", river_overall_summary_path.exists())
print("Category summary:", river_category_summary_path.exists())
print("Year summary:", river_year_summary_path.exists())

Saved river-flow summary tables:
/content/drive/MyDrive/STAT596/Project596_datafiles/tables/river_flow_overall_summary.csv
/content/drive/MyDrive/STAT596/Project596_datafiles/tables/river_flow_category_summary.csv
/content/drive/MyDrive/STAT596/Project596_datafiles/tables/river_flow_year_summary.csv

File exists check:
Overall summary: True
Category summary: True
Year summary: True


In [ ]:
# I had to restart the cleaning process once because of naming inconsistancy
# so now i will do a check on every processed/cleaned data


expected_final_columns = [
    "date",
    "start_datetime",
    "end_datetime",
    "interval_hours",
    "year",
    "month",
    "month_name",
    "site_id",
    "site_name",
    "parameter",
    "source",
    "discharge_m3s",
    "discharge_cfs",
    "discharge_mgd",
    "grade_code",
    "interpolation_type",
    "high_flow_75th",
    "high_flow_90th",
    "extreme_flow_95th",
    "high_flow_day",
    "flow_category",
    "flow_lag1_m3s",
    "flow_3day_mean_m3s",
    "flow_7day_mean_m3s",
    "high_flow_1day_window",
    "high_flow_3day_window",
    "high_flow_7day_window",
    "days_from_emergency_declaration",
    "post_emergency",
    "emergency_reference_window",
    "h2s_overlap_period",
    "analysis_period"
]

missing_expected_columns = [
    col for col in expected_final_columns
    if col not in river_daily.columns
]

extra_columns = [
    col for col in river_daily.columns
    if col not in expected_final_columns
]

print("Final QA for Tijuana River daily flow dataset")
print("=" * 60)

print("Rows:", len(river_daily))
print("Columns:", river_daily.shape[1])
print("Date range:", river_daily["date"].min().date(), "to", river_daily["date"].max().date())
print("Unique dates:", river_daily["date"].nunique())
print("Duplicate dates:", river_daily["date"].duplicated().sum())

print("\nMissing expected columns:")
print(missing_expected_columns)

print("\nExtra columns:")
print(extra_columns)

print("\nMissing values in key columns:")
key_columns = [
    "date",
    "discharge_m3s",
    "discharge_cfs",
    "discharge_mgd",
    "high_flow_day",
    "flow_3day_mean_m3s",
    "flow_7day_mean_m3s",
    "high_flow_1day_window",
    "high_flow_3day_window",
    "high_flow_7day_window"
]
print(river_daily[key_columns].isna().sum())

print("\nHigh-flow counts:")
print("High flow 75th:", river_daily["high_flow_75th"].sum())
print("High flow 90th:", river_daily["high_flow_90th"].sum())
print("Extreme flow 95th:", river_daily["extreme_flow_95th"].sum())
print("High-flow 1-day window:", river_daily["high_flow_1day_window"].sum())
print("High-flow 3-day window:", river_daily["high_flow_3day_window"].sum())
print("High-flow 7-day window:", river_daily["high_flow_7day_window"].sum())

print("\nFinal dataset preview:")
display(river_daily.head())
display(river_daily.tail())

Final QA for Tijuana River daily flow dataset
Rows: 1159
Columns: 32
Date range: 2023-01-01 to 2026-03-04
Unique dates: 1159
Duplicate dates: 0

Missing expected columns:
[]

Extra columns:
[]

Missing values in key columns:
date                     0
discharge_m3s            0
discharge_cfs            0
discharge_mgd            0
high_flow_day            0
flow_3day_mean_m3s       0
flow_7day_mean_m3s       0
high_flow_1day_window    0
high_flow_3day_window    0
high_flow_7day_window    0
dtype: int64

High-flow counts:
High flow 75th: 291
High flow 90th: 116
Extreme flow 95th: 58
High-flow 1-day window: 327
High-flow 3-day window: 383
High-flow 7-day window: 464

Final dataset preview:


,date,start_datetime,end_datetime,interval_hours,year,month,month_name,site_id,site_name,parameter,...,flow_3day_mean_m3s,flow_7day_mean_m3s,high_flow_1day_window,high_flow_3day_window,high_flow_7day_window,days_from_emergency_declaration,post_emergency,emergency_reference_window,h2s_overlap_period,analysis_period
0,2023-01-01,2023-01-01,2023-01-02,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,78.300000,78.3000,1,1,1,-177,0,0,0,Before emergency reference window
1,2023-01-02,2023-01-02,2023-01-03,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,43.880000,43.8800,1,1,1,-176,0,0,0,Before emergency reference window
2,2023-01-03,2023-01-03,2023-01-04,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,30.950000,30.9500,1,1,1,-175,0,0,0,Before emergency reference window
3,2023-01-04,2023-01-04,2023-01-05,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,6.023333,24.0925,1,1,1,-174,0,0,0,Before emergency reference window
4,2023-01-05,2023-01-05,2023-01-06,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,4.123333,20.0260,1,1,1,-173,0,0,0,Before emergency reference window


,date,start_datetime,end_datetime,interval_hours,year,month,month_name,site_id,site_name,parameter,...,flow_3day_mean_m3s,flow_7day_mean_m3s,high_flow_1day_window,high_flow_3day_window,high_flow_7day_window,days_from_emergency_declaration,post_emergency,emergency_reference_window,h2s_overlap_period,analysis_period
1154,2026-02-28,2026-02-28,2026-03-01,24.0,2026,2,February,11013300,Tijuana River At International Boundary,Discharge,...,1.856667,2.052857,0,0,1,977,1,0,1,After emergency reference window
1155,2026-03-01,2026-03-01,2026-03-02,24.0,2026,3,March,11013300,Tijuana River At International Boundary,Discharge,...,1.866667,1.961429,0,0,0,978,1,0,1,After emergency reference window
1156,2026-03-02,2026-03-02,2026-03-03,24.0,2026,3,March,11013300,Tijuana River At International Boundary,Discharge,...,1.850000,1.901429,0,0,0,979,1,0,1,After emergency reference window
1157,2026-03-03,2026-03-03,2026-03-04,24.0,2026,3,March,11013300,Tijuana River At International Boundary,Discharge,...,1.826667,1.857143,0,0,0,980,1,0,1,After emergency reference window
1158,2026-03-04,2026-03-04,2026-03-05,24.0,2026,3,March,11013300,Tijuana River At International Boundary,Discharge,...,1.786667,1.825714,0,0,0,981,1,0,1,After emergency reference window


In [ ]:
# now we save

river_processed_path = DATA_PROCESSED / "tijuana_river_daily_flow_2023_2026.csv"

river_daily.to_csv(river_processed_path, index=False)

print("Saved processed river-flow file:")
print(river_processed_path)

print("\nFile exists:", river_processed_path.exists())

# Reload saved file to confirm it opens correctly
river_check = pd.read_csv(river_processed_path)

print("\nReloaded file QA:")
print("Rows:", len(river_check))
print("Columns:", river_check.shape[1])
print("Date range:", river_check["date"].min(), "to", river_check["date"].max())
print("Missing discharge_m3s:", river_check["discharge_m3s"].isna().sum())

display(river_check.head())

Saved processed river-flow file:
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/tijuana_river_daily_flow_2023_2026.csv

File exists: True

Reloaded file QA:
Rows: 1159
Columns: 32
Date range: 2023-01-01 to 2026-03-04
Missing discharge_m3s: 0


,date,start_datetime,end_datetime,interval_hours,year,month,month_name,site_id,site_name,parameter,...,flow_3day_mean_m3s,flow_7day_mean_m3s,high_flow_1day_window,high_flow_3day_window,high_flow_7day_window,days_from_emergency_declaration,post_emergency,emergency_reference_window,h2s_overlap_period,analysis_period
0,2023-01-01,2023-01-01,2023-01-02,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,78.300000,78.3000,1,1,1,-177,0,0,0,Before emergency reference window
1,2023-01-02,2023-01-02,2023-01-03,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,43.880000,43.8800,1,1,1,-176,0,0,0,Before emergency reference window
2,2023-01-03,2023-01-03,2023-01-04,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,30.950000,30.9500,1,1,1,-175,0,0,0,Before emergency reference window
3,2023-01-04,2023-01-04,2023-01-05,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,6.023333,24.0925,1,1,1,-174,0,0,0,Before emergency reference window
4,2023-01-05,2023-01-05,2023-01-06,24.0,2023,1,January,11013300,Tijuana River At International Boundary,Discharge,...,4.123333,20.0260,1,1,1,-173,0,0,0,Before emergency reference window


In [ ]:
# checking if the file got saved
from pathlib import Path

NOTEBOOKS = Path("/content/drive/MyDrive/STAT596/Project596_datafiles/notebooks")

print("Notebook folder exists:", NOTEBOOKS.exists())

print("\nNotebook files:")
for file in NOTEBOOKS.glob("*.ipynb"):
    print(file.name)

Notebook folder exists: True

Notebook files:
03_process_tijuana_river_flow.ipynb
